<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Calculate Course Similarity using BoW Features**


Estimated time needed: **45** minutes


## **Introduction**
Similarity measurement is an important part of content-based recommendation systems.
The basic idea is that if two items have similar features, users who like one item may also be interested in the other.

To compute similarity between courses, we first need to represent each course as a numerical feature vector. In previous labs, we converted course descriptions into feature vectors using the Bag-of-Words (BoW) method. Although genre information is available, genre labels are limited and cannot fully describe the detailed content of each course. BoW features extracted from course descriptions provide richer information and allow more accurate similarity calculations.

Once each course is represented as a BoW feature vector, we can apply common similarity measurements such as `cosine similarity`, `Jaccard index`, or `Euclidean distance` to compute how similar two courses are. These similarity scores can then be used to support content-based recommendation.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_2/images/course_sim.png)


In this lab, we will use BoW feature vectors generated from course descriptions and calculate similarity between courses to support content-based recommendation.

## **Objectives**


* Calculate the similarity between any two courses using BoW feature vectors


----


### **Prepare and setup lab environment**


First, install and import required libraries:


In [ ]:
#!pip install nltk
#!pip install gensim
#!pip install scipy==1.10
#!pip install pandas
#!pip install matplotlib
#!pip install seaborn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import gensim
import pandas as pd
import nltk as nltk

from scipy.spatial.distance import cosine
from sklearn.metrics import jaccard_score
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk import ngrams
from gensim import corpora

%matplotlib inline

In [ ]:
# also set a random state
rs = 123

### **Create Bag-of-Words vectors for two example courses**


Suppose we have two simple example courses:


In [ ]:
course1 = "machine learning for everyone"

In [ ]:
course2 = "machine learning for beginners"

Next we can quickly tokenize them using the split() method (or using `word_tokenize()` method provided in `nltk` as we did in the previous lab).


In [ ]:
tokens = set(course1.split() + course2.split())

In [ ]:
tokens = list(tokens)
tokens

then generate BoW features (token counts) for these two courses (or using `tokens_dict.doc2bow()` method provided in `nltk`, similar to what we did in the previous lab).


In [ ]:
def generate_sparse_bow(course):
    """
    Generate a sparse bag-of-words (BoW) representation for a given course.

    Parameters:
    course (str): The input course text to generate the BoW representation for.

    Returns:
    list: A sparse BoW representation where each element corresponds to the presence (1) or absence (0)
    of a word in the input course text.
    """

    # Initialize an empty list to store the BoW vector
    bow_vector = []

    # Tokenize the course text by splitting it into words
    words = course.split()

    # Iterate through all unique words (tokens) in the course
    for token in tokens:
        # Check if the token is present in the course text
        if token in words:
            # If the token is present, append 1 to the BoW vector
            bow_vector.append(1)
        else:
            # If the token is not present, append 0 to the BoW vector
            bow_vector.append(0)

    # Return the sparse BoW vector
    return bow_vector


In [ ]:
bow1 = generate_sparse_bow(course1)
bow1

In [ ]:
bow2 = generate_sparse_bow(course2)
bow2

From the above cell outputs, we can see the two vectors are very similar. Only two dimensions are different.


### **Calculate the consine similarity between two example courses**

In [ ]:
cos_sim = 1 - cosine(bow1, bow2)

In [ ]:
print(f"The cosine similarity between course `{course1}` and course `{course2}` is {round(cos_sim, 2) * 100}%")

### **Calculate the  Jaccard index between two example courses**

_Practice: Try other similarity measurements such as Euclidean Distance or Jaccard index._


In [ ]:
# Calculate Jaccard similarity between pairs of documents using built-in function
jaccard_scores = jaccard_score(bow1, bow2)
# Print Jaccard similarity scores
print(f"The Jaccard similarity between course `{course1}` and course `{course2}` is {round(jaccard_scores, 2) * 100}%")


### **Calculate the Euclidean distance between two example courses**

In [ ]:
# Calculate Euclidean distance between pairs of documents using built-in function
euclidean_distances = np.linalg.norm(np.array(bow1) - np.array(bow2))

# Print Euclidean distance
print(f"The Euclidean distance between course `{course1}` and course `{course2}` is {round(euclidean_distances, 2) }")


For Example: Euclidean distance between 2 points $p$ and $q$ can be summarized by this equation: $d(p,q)={\sqrt {(p_{1}-q_{1})^{2}+(p_{2}-q_{2})^{2}+(p_{3}-q_{3})^{2}}}$. You can use `euclidean(p,q)` function from ```scipy``` package to calculate it. 


### **TASK: Find similar courses to the course `Machine Learning with Python`**


Now you have learned how to calculate cosine similarity between two sample BoW feature vectors. Let's work on some real course BoW feature vectors.


In [ ]:
# Load the BoW features as Pandas dataframe
bows_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/courses_bows.csv"
bows_df = pd.read_csv(bows_url)
bows_df = bows_df[['doc_id', 'token', 'bow']]

In [ ]:
bows_df.head(10)

The `bows_df` dataframe contains the BoW features vectors for each course, in a vertical and dense format. It has three columns `doc_id` represents the course id, `token` represents the token value, and `bow` represents the BoW value (token count).


Then, let's load another course content dataset which contains the course title and description:


In [ ]:
# Load the course dataframe
course_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv"
course_df = pd.read_csv(course_url)

In [ ]:
course_df.head(10)

Given course ID `ML0101ENv3`, let's find out its title and description:


In [ ]:
course_df[course_df['COURSE_ID'] == 'ML0101ENv3']

We can see it is a machine learning with Python course so we can expect any machine learning or Python related courses would be similar.


Then, let's print its associated BoW features:


In [ ]:
ml_course = bows_df[bows_df['doc_id'] == 'ML0101ENv3']
ml_course

We can see the BoW feature vector is in vertical format but normally feature vectors are in horizontal format. One way to transpose the feature vector from vertical to horizontal is to use the Pandas `pivot()` method:


In [ ]:
ml_courseT = ml_course.pivot(index=['doc_id'], columns='token').reset_index(level=[0])
ml_courseT

To compare the BoWs of any two courses, which normally have a different set of tokens, we need to create a union token set and then transpose them. We have provided a method called `pivot_two_bows` as follows:


In [ ]:
def pivot_two_bows(basedoc, comparedoc):
    """
    Pivot two bag-of-words (BoW) representations for comparison.

    Parameters:
    basedoc (DataFrame): DataFrame containing the bag-of-words representation for the base document.
    comparedoc (DataFrame): DataFrame containing the bag-of-words representation for the document to compare.

    Returns:
    DataFrame: A DataFrame with pivoted BoW representations for the base and compared documents,
    facilitating direct comparison of word occurrences between the two documents.
    """

    # Create copies of the input DataFrames to avoid modifying the originals
    base = basedoc.copy()
    base['type'] = 'base'  # Add a 'type' column indicating base document
    compare = comparedoc.copy()
    compare['type'] = 'compare'  # Add a 'type' column indicating compared document

    # Concatenate the two DataFrames vertically
    join = pd.concat([base, compare])

    # Pivot the concatenated DataFrame based on 'doc_id' and 'type', with words as columns
    joinT = join.pivot(index=['doc_id', 'type'], columns='token').fillna(0).reset_index(level=[0, 1])

    # Assign meaningful column names to the pivoted DataFrame
    joinT.columns = ['doc_id', 'type'] + [t[1] for t in joinT.columns][2:]

    # Return the pivoted DataFrame for comparison
    return joinT


In [ ]:
course1 = bows_df[bows_df['doc_id'] == 'ML0151EN']
course2 = bows_df[bows_df['doc_id'] == 'ML0101ENv3']

In [ ]:
bow_vectors = pivot_two_bows(course1, course2)
bow_vectors

Similarly, we can use the cosine method to calculate their similarity:


In [ ]:
similarity = 1 - cosine(bow_vectors.iloc[0, 2:], bow_vectors.iloc[1, 2:])
similarity

Now it's your turn to perform a task of finding all courses similar to the course `Machine Learning with Python`:


In [ ]:
course_df[course_df['COURSE_ID'] == 'ML0101ENv3']

You can set a similarity threshold such as 0.5 to determine if two courses are similar enough.


_TODO: Find courses which are similar to course `Machine Learning with Python (ML0101ENv3)`, you also need to show the title and descriptions of those courses._


In [ ]:
course1 = bows_df[bows_df['doc_id'] != 'ML0101ENv3']
course_df['COURSE_ID']

In [ ]:
## For each course other than ML0101ENv3, use pivot_course_rows to convert it with course ML0101ENv3 into horizontal two BoW feature vectors
## Then use the cosine method to calculate the similarity
## Report all courses with similarities larger than a specific threshold (such as 0.5)
course2 = bows_df[bows_df['doc_id'] == 'ML0101ENv3']

sim_result = []
for course in course_df['COURSE_ID']:
    if course == 'ML0101ENv3':
        continue
        
    course1 = bows_df[bows_df['doc_id'] == course]
    bow_vectors = pivot_two_bows(course1, course2)

    similarity = 1 - cosine(
        bow_vectors.iloc[0, 2:].astype(float),
        bow_vectors.iloc[1, 2:].astype(float)
    )
    
    new_row = {"course": course, "cos_sim_score": similarity}
    sim_result.append(new_row)   

sim_result = pd.DataFrame(sim_result)

sim_result[sim_result["cos_sim_score"]>=0.5]
        

<details>
    <summary>Click here for Hints</summary>
    
You can use `bows_df[bows_df['doc_id'] == 'ML0101ENv3']` to find 'ML0101ENv3' course bow. Then in a similar matter you can find bows for each course_id that's not 'ML0101ENv3'. Then you can join 2 bows by using predefined `pivot_two_bows` function and calculate the similarity as we just did using the cosine method. Print the course ids with similarity>0.5 
</details>


## **Summary**


In this lab, we used cosine and course BoW features to calculate the similarities among courses. Such similarity measurement is the core of many content-based recommender systems, which we will learn and practice in the later labs.


## **Authors**


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/), Su Wu


Copyright © 2021 IBM Corporation. All rights reserved.
